## Imports

In [1]:
from langchain_groq import ChatGroq
from langchain.schema import HumanMessage

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

def get_llm_llama():
    return ChatGroq(
        groq_api_key=GROQ_API_KEY,
        model_name="llama-3.1-8b-instant",
        temperature=0.0,
        max_tokens=512
    )

***Split large text into chunks***

In [3]:
def split_text(text, chunk_size=1500):
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]


# Summarize each chunk
def summarize_chunk(llm, chunk):
    prompt = f"""
    Summarize the following movie reviews into a short paragraph.

    Reviews:
    {chunk}
    """

    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content

# Final summarization
def summarize_reviews(full_text):
    llm = get_llm_llama()
    chunks = split_text(full_text)

    chunk_summaries = [summarize_chunk(llm, chunk) for chunk in chunks]

    final_prompt = f"""
    Combine the following summaries into ONE clear and concise paragraph summary of the movie reviews.
    Do NOT use bullet points or headings.

    Summaries:
    {" ".join(chunk_summaries)}
    """

    final_summary = llm.invoke([HumanMessage(content=final_prompt)])

    return final_summary.content


# Run
# with open("Text Files/avatar_review.txt", "r", encoding="utf-8") as f:
#     full_text = f.read()

full_text = input()

final_output = summarize_reviews(full_text)

# ONLY final summary
print("\n========================\n")
print(final_output)

1. Sometimes I just need to see the start. Or end. Or a trailer. Or the music and theme from Hans Zimmer. Or the whole movie. Just to feel that thing, I only get from this movie. That the earth, space and time are something special, mystical. I never forget the first time I saw this movie, in an IMAX theatre in 2014. I was struck by it. Totally got me. And it stil does, 7 years later. This is the best movie ever made for me. Because of the feeling it gives me, no other movie can. So hard to get all of this emotion in only one movie. Brilliant. 2. After watching this insane movie in the theatres back in 2014 I swore to god I will wait 5 years to watch it again so I get to forget it and experince the insanity it has again This without doubt is THE BEST MOVIE EVER MADE 3. A lot has been said and written about Interstellar. You can obviously take apart any movie that is out there. You'll either love this one or you won't. I kind of would have loved to have watched this on an IMAX screen, t

## Summarization Evaluation

In [4]:
reference_summary = """Interstellar is a very popular movie directed by Christopher Nolan. 
                        Many viewers feel it is a visually amazing and emotionally powerful film. 
                        The story, music, acting, and cinematography are highly appreciated. 
                        Background score and the space visuals give a strong emotional experience to the audience. 
                        Most of consider it one of the best movies ever made because of its unique concept and emotional depth. 
                        Some viewers have different opinions about the ending, 
                        the movie is loved for its storytelling, scientific ideas, and memorable experience."""


In [5]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

generated_summary = final_output

scores = scorer.score(reference_summary, generated_summary)

print(scores)

{'rouge1': Score(precision=0.37719298245614036, recall=0.5119047619047619, fmeasure=0.4343434343434343), 'rouge2': Score(precision=0.1592920353982301, recall=0.21686746987951808, fmeasure=0.1836734693877551), 'rougeL': Score(precision=0.2543859649122807, recall=0.34523809523809523, fmeasure=0.29292929292929293)}


In [6]:
# !pip install rouge_score

## Generic Summary Baseline

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def generic_baseline_summary(text, n_sentences=3):
    #  split into sentences
    sentences = text.split(".")
    sentences = [s.strip() for s in sentences if len(s.strip()) > 0]

    if len(sentences) == 0:
        return ""

    # TF-IDF scoring
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(sentences)

    #  sentence importance score
    scores = np.array(tfidf_matrix.sum(axis=1)).flatten()

    # Step 4: select top sentences
    top_indices = scores.argsort()[-n_sentences:]
    top_indices = sorted(top_indices)

    # Step 5: build summary
    summary = " ".join([sentences[i] for i in top_indices])

    return summary

In [8]:
baseline_summary = generic_baseline_summary(full_text)

print("Baseline Summary:\n")
print(baseline_summary)

Baseline Summary:

After watching this insane movie in the theatres back in 2014 I swore to god I will wait 5 years to watch it again so I get to forget it and experince the insanity it has again This without doubt is THE BEST MOVIE EVER MADE 3 Going out and saying this will be considered a classic, might not be too far stretched, but you still can never predict those things The deserved love the movie gets on IMDb and other places would be an indicator that this will ring true though


In [9]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

reference_summary = baseline_summary

generated_summary = final_output

scores = scorer.score(reference_summary, generated_summary)

print(scores)

{'rouge1': Score(precision=0.23684210526315788, recall=0.29347826086956524, fmeasure=0.2621359223300971), 'rouge2': Score(precision=0.07964601769911504, recall=0.0989010989010989, fmeasure=0.08823529411764706), 'rougeL': Score(precision=0.12280701754385964, recall=0.15217391304347827, fmeasure=0.13592233009708737)}


---

## LLM Generated Summary vs Reference

| Metric  | Precision | Recall | F1 Score |
| ------- | --------- | ------ | -------- |
| ROUGE-1 | 0.38      | 0.51   | **0.43** |
| ROUGE-2 | 0.16      | 0.22   | **0.18** |
| ROUGE-L | 0.25      | 0.35   | **0.29** |

---

##  Baseline Summary vs Reference

| Metric  | Precision | Recall | F1 Score |
| ------- | --------- | ------ | -------- |
| ROUGE-1 | 0.24      | 0.29   | **0.26** |
| ROUGE-2 | 0.08      | 0.10   | **0.09** |
| ROUGE-L | 0.12      | 0.15   | **0.13** |

---

# Direct Comparison 

| Metric  | LLM Model | Baseline | Improvement|
| ------- | --------- | -------- | -----------|
| ROUGE-1 | **0.43**  | 0.26     |  +65%     |
| ROUGE-2 | **0.18**  | 0.09     |  +100%    |
| ROUGE-L | **0.29**  | 0.13     |  +123%    |

---

# Interpretation (Simple Explanation)

##  ROUGE-1 (Word Overlap)

* LLM: **0.43**
* Baseline: **0.26**

Your LLM captures **more important words** from the reference.

Meaning:

* Better understanding of key terms like *film, story, emotion*

---

##  ROUGE-2 (Phrase Overlap)

* LLM: **0.18**
* Baseline: **0.09**

LLM is **twice as good** in matching meaningful phrases.

✔ Meaning:

* It forms better natural expressions
* Not just random sentence picking

---

##  ROUGE-L (Structure Similarity)

* LLM: **0.29**
* Baseline: **0.13**

LLM preserves **sentence flow and meaning better**

✔ Meaning:

* More coherent summary
* Better readability

---

# Final Conclusion

 Your LLM clearly outperforms the baseline in ALL metrics.

###  Key Takeaways:

* LLM captures **more content** (higher recall)
* LLM produces **better phrases** (ROUGE-2)
* LLM maintains **better structure** (ROUGE-L)
* Baseline is weak because it just selects sentences

